# Sprint 2b: MLP Hyperparameter Search

Thin runner for MLP-only experiments on cached frozen features. Feature extraction should already be complete.

Run `00_colab_setup.ipynb` first for the dataset, then `02_extract_frozen_features.ipynb` to create the frozen feature caches.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/KasimDeliaci/dl-midterm-project.git'
REPO_ROOT = Path('/content/dl-assignment')
BUNDLE_CANDIDATES = [
    Path('/content/drive/MyDrive/ham10000_colab_bundle.tar'),
    Path('/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar'),
    Path('/content/drive/MyDrive/dl-assignment/ham10000_colab_bundle.tar'),
]
DRIVE_ARTIFACTS = Path('/content/drive/MyDrive/dl-midterm-artifacts')

drive.mount('/content/drive', force_remount=True)
DRIVE_BUNDLE = next((path for path in BUNDLE_CANDIDATES if path.exists()), BUNDLE_CANDIDATES[0])

if not (REPO_ROOT / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
print('Repo root:', Path.cwd())
print('Drive bundle exists:', DRIVE_BUNDLE.exists(), DRIVE_BUNDLE)
print('Checked bundle candidates:')
for path in BUNDLE_CANDIDATES:
    print(' -', path, path.exists())

In [ ]:
required_paths = [
    REPO_ROOT / 'data/raw/ham10000',
    REPO_ROOT / 'data/metadata/HAM10000_metadata.csv',
    REPO_ROOT / 'data/splits/train.csv',
    REPO_ROOT / 'data/splits/val.csv',
    REPO_ROOT / 'data/splits/test.csv',
]

if not all(path.exists() for path in required_paths):
    if not DRIVE_BUNDLE.exists():
        raise FileNotFoundError(f'Missing Drive bundle: {DRIVE_BUNDLE}')
    subprocess.run(['tar', '--warning=no-unknown-keyword', '-xf', str(DRIVE_BUNDLE), '-C', str(REPO_ROOT)], check=True)

missing = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise RuntimeError(f'Missing required dataset paths after extraction: {missing}')

print('Dataset and preserved Sprint 1 split CSVs are available.')
for path in required_paths:
    print(path.relative_to(REPO_ROOT))

from pathlib import Path
import shutil

feature_root = Path('artifacts/features/ham10000/frozen')
drive_feature_root = DRIVE_ARTIFACTS / feature_root
if not feature_root.exists() and drive_feature_root.exists():
    feature_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(drive_feature_root, feature_root, dirs_exist_ok=True)
    print('Restored frozen feature cache from Drive:', drive_feature_root)

required_cache_files = [
    feature_root / backbone / split
    for backbone in ['resnet50', 'mobilenet_v2', 'efficientnet_b0']
    for split in ['train.pt', 'val.pt', 'test.pt']
]
missing = [str(path) for path in required_cache_files if not path.exists()]
if missing:
    raise RuntimeError('Missing frozen feature caches. Run notebooks/02_extract_frozen_features.ipynb first. Missing example: ' + missing[0])

print('Frozen feature caches are available.')

In [ ]:
!python -m pip install -q uv
!uv sync

Run the focused search. Each run gets a separate folder under `artifacts/runs/mlp_hparam_search/<search_id>/`.

In [ ]:
RUN_FULL_SEARCH = False

if RUN_FULL_SEARCH:
    import subprocess
    subprocess.run([
        'uv', 'run', 'python', 'scripts/run_mlp_hyperparam_search.py',
        '--config', 'configs/experiments/mlp_hyperparam_search.yaml',
        '--default-config', 'configs/default.yaml',
        '--dataset-config', 'configs/dataset/selected_dataset.yaml',
    ], check=True)
else:
    print('Skipped full MLP hyperparameter search. Set RUN_FULL_SEARCH = True for the full grid.')

For a quick Colab smoke test, limit the grid.

In [ ]:
%%bash
uv run python scripts/run_mlp_hyperparam_search.py \
  --config configs/experiments/mlp_hyperparam_search.yaml \
  --backbones resnet50 \
  --candidates cw_adamw_d03 nocw_adamw_d03 \
  --epochs 3